<a href="https://colab.research.google.com/github/nejumi/weave-initial-course/blob/main/notebooks/02_practice/02_langgraph_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05. LangGraph エージェント + Weave トレース

LangGraph で ReAct パターンのエージェントを実装し、Weave でマルチステップのトレースを可視化します。

## 学習内容
- LangGraph の基本（StateGraph、ノード、エッジ）
- ReAct パターン（Reasoning + Acting）
- `@weave.op` による各ノードのトレース
- Weave UI でのグラフ実行の可視化


In [ ]:
# ローカル環境: uv sync --all-extras で一括インストール可能
# Colab: 以下のセルで個別インストール
!pip install -q uv
!uv pip install --system -q \
  "weave>=0.51.0" \
  "wandb>=0.19.0" \
  "openai>=1.0.0" \
  "python-dotenv>=1.0.0" \
  "langgraph>=0.2.0" \
  "langchain-openai>=0.2.0"

In [ ]:
import os
try:
    import google.colab
    IN_COLAB = True
    from google.colab import userdata
    # WANDB_BASE_URL: 値がある場合のみセット（空 / 未設定 → SaaS デフォルト）
    _base_url = (userdata.get("WANDB_BASE_URL") or "").strip()
    if _base_url:
        os.environ["WANDB_BASE_URL"] = _base_url
    os.environ.setdefault("WANDB_API_KEY",  userdata.get("WANDB_API_KEY"))
    os.environ.setdefault("OPENAI_API_KEY", userdata.get("OPENAI_API_KEY"))
    os.environ.setdefault("WANDB_ENTITY",   userdata.get("WANDB_ENTITY"))
    os.environ.setdefault("WANDB_PROJECT",  userdata.get("WANDB_PROJECT") or "weave-course")
except ImportError:
    IN_COLAB = False
    from dotenv import load_dotenv; load_dotenv()
    # ローカル: .env に空欄で書かれていた場合も削除
    if not os.environ.get("WANDB_BASE_URL", "").strip():
        os.environ.pop("WANDB_BASE_URL", None)

ENTITY  = os.environ["WANDB_ENTITY"]
PROJECT = os.environ.get("WANDB_PROJECT", "weave-course")
_target = os.environ.get("WANDB_BASE_URL", "https://api.wandb.ai (SaaS)")
print(f"Entity : {ENTITY}")
print(f"Project: {PROJECT}")
print(f"W&B URL: {_target}")


In [ ]:
import json
import weave
from openai import OpenAI

client_weave = weave.init(f"{ENTITY}/{PROJECT}")


## 1. 状態定義とツール

In [ ]:
import math
from typing import TypedDict, Annotated
import operator

# ── グラフの状態 ──────────────────────────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]
    tool_calls_count: int

# ── ツール（@weave.op でトレース） ───────────────────────────────────────────
@weave.op()
def tool_calculate(expression: str) -> str:
    try:
        result = eval(expression, {"__builtins__": {}}, {"math": math})
        return f"{result}"
    except Exception as e:
        return f"Error: {e}"

@weave.op()
def tool_lookup(topic: str) -> str:
    kb = {
        "weave":    "W&B Weave is an LLM observability platform.",
        "langgraph": "LangGraph is a library for building stateful, multi-step LLM applications.",
        "grpo":     "GRPO is Group Relative Policy Optimization, used in LLM reinforcement learning.",
        "wandb":    "Weights & Biases (W&B) provides experiment tracking and MLOps tools.",
    }
    key = topic.lower()
    return next((v for k, v in kb.items() if k in key), f"No information found for: {topic}")

@weave.op()
def tool_weather(city: str) -> str:
    data = {"tokyo": "22°C Sunny", "osaka": "24°C Cloudy", "sapporo": "12°C Rainy"}
    return data.get(city.lower(), "20°C Unknown")

TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "calculate", "description": "Evaluate math expression (Python syntax)",
        "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]},
    }},
    {"type": "function", "function": {
        "name": "lookup", "description": "Look up information about a topic",
        "parameters": {"type": "object", "properties": {"topic": {"type": "string"}}, "required": ["topic"]},
    }},
    {"type": "function", "function": {
        "name": "weather", "description": "Get weather for a city",
        "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]},
    }},
]

TOOL_MAP = {"calculate": tool_calculate, "lookup": tool_lookup, "weather": tool_weather}
print("ツール定義完了")


## 2. LangGraph グラフ定義

In [ ]:
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# LangGraph ノードを @weave.op でラップ
@weave.op()
def agent_node(state: AgentState) -> AgentState:
    """LLM が次の行動（ツール呼び出し or 最終回答）を決定するノード"""
    oai = OpenAI()
    resp = oai.chat.completions.create(
        model="gpt-4o-mini",
        messages=state["messages"],
        tools=TOOLS_SCHEMA,
        tool_choice="auto",
    )
    msg = resp.choices[0].message
    return {
        "messages": [{"role": "assistant", "content": msg.content, "tool_calls": msg.tool_calls}],
        "tool_calls_count": state["tool_calls_count"],
    }

@weave.op()
def tool_node(state: AgentState) -> AgentState:
    """ツールを実行して結果を返すノード"""
    last_msg = state["messages"][-1]
    results = []
    for tc in (last_msg.get("tool_calls") or []):
        func_name = tc.function.name
        args = json.loads(tc.function.arguments)
        result = TOOL_MAP[func_name](**args)
        results.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "content": str(result),
        })
    return {
        "messages": results,
        "tool_calls_count": state["tool_calls_count"] + len(results),
    }

def should_continue(state: AgentState) -> str:
    """ツール呼び出しがあれば tools ノードへ、なければ終了"""
    last_msg = state["messages"][-1]
    if last_msg.get("tool_calls"):
        return "tools"
    return END

# グラフ構築
graph_builder = StateGraph(AgentState)
graph_builder.add_node("agent", agent_node)
graph_builder.add_node("tools", tool_node)
graph_builder.set_entry_point("agent")
graph_builder.add_conditional_edges("agent", should_continue)
graph_builder.add_edge("tools", "agent")
graph = graph_builder.compile()
print("グラフ構築完了")


## 3. エージェント実行

In [ ]:
@weave.op()
def run_langgraph_agent(question: str) -> str:
    """LangGraph エージェントを実行して最終回答を返す"""
    initial_state = {
        "messages": [
            {"role": "system", "content": "You are a helpful assistant. Use tools when needed. Answer in Japanese."},
            {"role": "user",   "content": question},
        ],
        "tool_calls_count": 0,
    }
    final_state = graph.invoke(initial_state)
    last = final_state["messages"][-1]
    return last.get("content", "") if isinstance(last, dict) else str(last)

# テスト実行
questions = [
    "東京の天気と気温の2乗を教えてください",
    "LangGraphとWeaveについて調べて、共通点を教えてください",
    "2の8乗 + 3の4乗を計算してください",
]

for q in questions:
    print()
    print(f"質問: {q}")
    answer = run_langgraph_agent(q)
    print(f"回答: {answer}")


## 4. Weave UI での確認ポイント

- `run_langgraph_agent` が root トレース
- `agent_node` → `tool_node` → `agent_node` のループが階層表示
- 各ツール（`tool_calculate` など）が `tool_node` の子スパン
- **何ステップで収束したか**がトレースから読み取れる


In [ ]:
print(f"Weave UI: https://wandb.ai/{ENTITY}/{PROJECT}/weave/traces")


## まとめ

次: **03_advanced/01_wandb_models_registry.ipynb** → W&B Models Registry